# End-to-End Medical X-Ray Dataset Pipeline

1. **OCR Extraction** (docTr)
2. **Data Cleaning & Rescue** (Regex, EasyOCR, Label Normalization)
3. **Hospital Matching** (Date blocking, SequenceMatcher)
4. **Auto Labeling** (Rule-based keywords mapping)
5. **Feature Engineering & Stratified Split** (Age, Sex, Study Grouping)
6. **Balance Data & Resize to SSD** (Drop useless classes, Age filter, 256x256 resizing)

In [1]:
# Tắt cảnh báo (warnings) để log sạch sẽ
import warnings
warnings.filterwarnings('ignore')

# Import class Pipeline
from pipeline import MedicalXRayPipeline

In [2]:
# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN / CONFIGURATION
# Thay đổi các đường dẫn này theo server thực tế
# ==========================================

CONFIG = {
    "root_img_dir": "F:/VCXR-Test/Data",                          # Thư mục gốc chứa ảnh X-quang mới / Raw Images
    "hospital_csv_path": "F:/VCXR-Test/Result/ketqua.csv",        # File excel từ phần mềm bệnh viện / HIS data
    "output_dir": "F:/VCXR-Test/Output",                          # Nơi lưu CSV Checkpoint trung gian / Checkpoint folder
    "ssd_resize_dir": "C:/VCXR",                                  # Nơi lưu ảnh Resize (SSD tốc độ cao) / SSD Output
    "gpu_batch_size": 128,                                        # Tùy biến theo VRAM của máy (12GB -> 128, 24GB -> 256)
    "img_size": 256,                                              # Kích thước resize AI
    "ocr_checkpoint_every": 500                                   # Lưu nháp sau mỗi 200 ảnh
}

print("Cấu hình thành công!")

Cấu hình thành công!


In [3]:
# Khởi tạo Pipeline / Init Pipeline
xray_pipeline = MedicalXRayPipeline(
    root_img_dir=CONFIG['root_img_dir'],
    hospital_csv_path=CONFIG['hospital_csv_path'],
    output_dir=CONFIG['output_dir'],
    ssd_resize_dir=CONFIG['ssd_resize_dir'],
    gpu_batch_size=CONFIG['gpu_batch_size'],
    img_size=CONFIG['img_size'],
    ocr_checkpoint_every=CONFIG['ocr_checkpoint_every']
)

### Bắt đầu chạy Pipeline
Mọi thao tác lưu nháp (Checkpoint) được tự động. Nếu vô tình tắt Kernel hoặc Crash, chỉ cần chạy lại Cell này, hệ thống sẽ tự động skip các bước đã thành công và Resume lại vị trí hiện tại.

In [4]:
xray_pipeline.run_pipeline()

2026-05-10 13:07:49,095 - INFO - 
############################################################
2026-05-10 13:07:49,097 - INFO - 🚀 BẮT ĐẦU END-TO-END MEDICAL X-RAY PIPELINE
2026-05-10 13:07:49,097 - INFO - ############################################################
2026-05-10 13:07:49,098 - INFO - 
2026-05-10 13:07:49,100 - INFO - === BƯỚC 1: BẮT ĐẦU TRÍCH XUẤT TEXT (OCR) ===
2026-05-10 13:07:49,100 - INFO - ============================================================
2026-05-10 13:07:49,182 - INFO - 📂 Tìm thấy 1350 ảnh mới cần xử lý.
2026-05-10 13:07:49,213 - INFO - 🧠 Đang sử dụng thiết bị: cuda
🚀 Processing Batches: 100%|██████████| 11/11 [03:04<00:00, 16.80s/it]
2026-05-10 13:10:55,609 - INFO - ✅ Hoàn tất BƯỚC 1. Đã xử lý 1350 ảnh.
INFO:MedicalXRayPipeline:✅ Hoàn tất BƯỚC 1. Đã xử lý 1350 ảnh.
2026-05-10 13:10:55,619 - INFO - 
INFO:MedicalXRayPipeline:
2026-05-10 13:10:55,621 - INFO - === BƯỚC 2: BẮT ĐẦU LÀM SẠCH VÀ CỨU DỮ LIỆU ===
INFO:MedicalXRayPipeline:=== BƯỚC 2: BẮT ĐẦU LÀM SẠ

### Phân tích sơ bộ Dataset đầu ra
*(Chạy Cell này sau khi Pipeline Success)*

In [7]:
import pandas as pd

try:
    df_train = pd.read_csv('F:/VCXR-Test/Output/final_dataset_train.csv')
    print(f"Tập Train hoàn chỉnh: {len(df_train)} records.")
    df_train.head()
except Exception as e:
    print("Chưa có file kết quả. Hãy chờ Pipeline chạy xong.")

df_train.head

Tập Train hoàn chỉnh: 431 records.


<bound method NDFrame.head of         ketqua_ID                                           filepath  \
0    2.400040e+09               ['C:/VCXR/train/2400040153.0_0.jpg']   
1    2.400044e+09               ['C:/VCXR/train/2400044436.0_0.jpg']   
2    2.400040e+09  ['C:/VCXR/train/2400039868.0_0.jpg', 'C:/VCXR/...   
3    2.400035e+09               ['C:/VCXR/train/2400035459.0_0.jpg']   
4    2.400042e+09  ['C:/VCXR/train/2400042128.0_0.jpg', 'C:/VCXR/...   
..            ...                                                ...   
426  2.400035e+09               ['C:/VCXR/train/2400035257.0_0.jpg']   
427  2.400038e+09  ['C:/VCXR/train/2400038017.0_0.jpg', 'C:/VCXR/...   
428  2.400044e+09  ['C:/VCXR/train/2400043586.0_0.jpg', 'C:/VCXR/...   
429  2.400038e+09               ['C:/VCXR/train/2400037693.0_0.jpg']   
430  2.400039e+09  ['C:/VCXR/train/2400039130.0_0.jpg', 'C:/VCXR/...   

                                       ketqua_Diagnose  xray_type  Age  Sex  \
0     J18.0-Viêm phế quản 